# Keras LSTM for IEEE-CIS Fraud Detection — per-UID sequence classifier

This notebook trains a Keras/TensorFlow LSTM on the IEEE-CIS fraud dataset, framed as
**per-UID sequence classification**:

> For each transaction, build a window of the user's previous K-1 transactions plus
> the current one, and predict the current transaction's `isFraud`.

## Why this framing (not the global-DT one)

A global `DT`-sorted window mostly mixes unrelated users. Measured on this dataset:

- In a global 30-row window, the median number of distinct UIDs is **29**.
- The next transaction's UID appears in the previous 30 global rows only **~9.1%** of the time.

So we group by UID first, then sort by `DT` inside each group. The model trains on
many independent per-UID sequences. This is panel/time-series data, not a single global
time series.

## Sparsity caveats (and why we use a small window)

Per-UID history is sparse:

- Median UID sequence length = **1**
- Only ~13.6% of UIDs have ≥5 transactions
- Only ~0.4% have ≥30 transactions
- **77.9% of test rows are cold-start by UID** (UID never seen in train)

So a window of 30 would be padded empty most of the time. We use a **small window
(default 5)** plus explicit time-gap features so the model can reason about both the
short history and the gaps between events.

## Setup expected

You should have run `Time_Series_Fraud_with_Magic.ipynb` up to the point where
`X_train_copy4` / `X_test_copy4` exist (UID built, time features added) and saved them:

```python
X_train_copy4.to_parquet('X_train_copy4.parquet')
X_test_copy4.to_parquet('X_test_copy4.parquet')
y_train.to_frame('isFraud').to_parquet('y_train.parquet')
```


## MPS / Apple Silicon GPU notes

This notebook is tuned to use the Metal GPU on your Apple Silicon Mac. To enable it,
install the Metal plugin once in your environment:

```bash
pip install tensorflow-macos tensorflow-metal
```

After that, TensorFlow will automatically pick up the Metal device. The code in the
next cell verifies the device is found, sets memory-growth so it doesn't pre-allocate
the whole GPU, and bumps the batch size — Apple Silicon's unified memory likes large
batches, and the per-step overhead of small batches dominates wall-clock time.

A few specific design choices below are MPS-aware:

- **`FAST_PATH = True`** — uses a model variant **without `Masking` and without
  `dropout=` inside the `LSTM(...)` call**. Both of those force a step-by-step
  Python loop in the recurrent layer instead of the fused Metal kernel; turning them
  off can be **3–10× faster** on M-series GPUs. Set `FAST_PATH = False` if you want
  the original (slightly more rigorous) variant for thesis comparison.
- **`tf.data` pipeline with `prefetch(AUTOTUNE)`** — overlaps host-side batching
  with GPU compute. Because Apple Silicon shares CPU/GPU memory, the gain is smaller
  than on discrete GPUs but still real.
- **Larger batch size (default 4096)** — feel free to push to 8192 if you have a
  16GB+ Mac. With `WINDOW=5` and ~250 features, one batch of 4096 is ~20MB.
- **No mixed precision by default** — Metal supports `mixed_float16`, but LSTMs
  are notorious for numerical issues under fp16 and the speed gain is modest.
  There's a commented toggle below if you want to experiment.


In [1]:
import sys, os
print(sys.executable)
print(os.environ.get("CONDA_DEFAULT_ENV"))

/Users/hovietbach/miniforge3/envs/fraud_tf_clean/bin/python
fraud_tf_clean


In [2]:
import numpy as np, pandas as pd, ml_dtypes, tensorflow as tf
print("numpy", np.__version__)
print("pandas", pd.__version__)
print("ml_dtypes", ml_dtypes.__version__)
print("tf", tf.__version__)
print("bfloat16 dtype:", np.dtype(ml_dtypes.bfloat16))
print("GPUs:", tf.config.list_physical_devices("GPU"))

TypeError: expected 0 arguments, got 1

In [3]:
# 0. Imports and config — MPS-aware
import os, gc, math, time, datetime, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input, Masking, LSTM, GRU, Dense, Dropout, BatchNormalization, Bidirectional
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.metrics import AUC

# ----- Configuration -----
DATA_DIR  = '/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/pkl_exported_files'
WINDOW    = 5
N_FOLDS   = 6
BATCH     = 4096          # larger batch — better GPU utilisation on Metal
EPOCHS    = 12
SEED      = 42
FAST_PATH = True          # see MPS notes above; turn off for the original variant
USE_MIXED_PRECISION = False   # experimental on Metal; LSTMs may NaN

tf.random.set_seed(SEED); np.random.seed(SEED)

# ----- Verify the Metal GPU is found and configure it -----
gpus = tf.config.list_physical_devices('GPU')
print('TF version:', tf.__version__)
print('Detected GPUs:', gpus)
if gpus:
    for g in gpus:
        try:
            tf.config.experimental.set_memory_growth(g, True)
            print(f'  memory_growth=True for {g.name}')
        except Exception as e:
            print(f'  could not set memory growth for {g.name}: {e}')
else:
    print('  WARNING: no GPU detected. Install tensorflow-metal:')
    print('     pip install tensorflow-macos tensorflow-metal')

if USE_MIXED_PRECISION:
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print('Mixed precision policy:', tf.keras.mixed_precision.global_policy())

AUTOTUNE = tf.data.AUTOTUNE


TF version: 2.16.2
Detected GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
  memory_growth=True for /physical_device:GPU:0


In [4]:
import tensorflow as tf
with tf.device('/GPU:0'):
    x = tf.random.normal([4096, 4096])
    y = tf.random.normal([4096, 4096])
    z = tf.matmul(x, y)
print(z.device)

/job:localhost/replica:0/task:0/device:GPU:0


2026-05-06 22:58:56.993681: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-05-06 22:58:56.995003: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-05-06 22:58:56.995023: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.33 GB
2026-05-06 22:58:56.997208: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-05-06 22:58:56.997229: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [7]:
import importlib, sys

sys.modules.setdefault("numpy._core", importlib.import_module("numpy.core"))
for m in ["numeric", "multiarray", "umath", "_multiarray_umath"]:
    try:
        sys.modules.setdefault(f"numpy._core.{m}", importlib.import_module(f"numpy.core.{m}"))
    except ModuleNotFoundError:
        pass

## 1. Load preprocessed checkpoints


In [9]:
X_train = pd.read_pickle(os.path.join(DATA_DIR, 'X_train_copy4.pkl'))
X_test  = pd.read_pickle(os.path.join(DATA_DIR, 'X_test_copy4.pkl'))
y_train = pd.read_pickle(os.path.join(DATA_DIR, 'y_train.pkl')).iloc[:, 0].astype('int8')

print('train rows :', len(X_train))
print('test  rows :', len(X_test))
print('train cols :', X_train.shape[1])
print('positive rate:', y_train.mean().round(4))


TypeError: issubclass() arg 1 must be a class

## 2. Feature selection

Drop housekeeping columns, the target, and known-bad columns from the original notebook's
exclusion list. Keep `uid` and `DT` aside — they're only needed for sorting / window-building,
not as model inputs.


In [ ]:
HOUSEKEEPING = {
    'TransactionDT', 'D6','D7','D8','D9','D12','D13','D14',
    'uid', 'day', 'DT', 'isFraud', 'oof',
    'C3','M5','id_08','id_33',
    'card4','id_07','id_14','id_21','id_30','id_32','id_34',
    *(f'id_{x}' for x in range(22, 28)),
}

feature_cols = [c for c in X_train.columns if c not in HOUSEKEEPING]
print('feature columns:', len(feature_cols))


## 3. Bridge train+test, then add time-gap features

**Train+test bridge.** We concatenate train and test into a single dataframe (with a
`__source__` flag), then compute time-gap features on the combined data and build
windows on the combined data. This is what lets a *test* transaction see its UID's
*train* history when the UID appears in both — about 22% of test rows.

This is safe with respect to leakage because:

1. Test rows are temporally **after** train rows in the IEEE dataset (May 2018 train
   ends, July 2018 test begins — there's even a 1-month gap), so when sorted by
   `(uid, DT)`, train rows of a UID always come before test rows.
2. The window is **strictly backward-looking** — for row `t`, we take rows `[t-window+1, t]`.
   So a train row never sees a test row in its window.
3. Inside training, we use `GroupKFold` on `DT_M`. Each fold's validation rows can see
   earlier-month rows in their window — which is correct, because at inference time
   on month 5 we'd have months 1–4 available.

**Time-gap features.** The LSTM does not automatically know how much real time elapsed
between two consecutive transactions of the same UID. We add three explicit features
(now correctly bridging the train/test boundary for known UIDs):

- `delta_seconds_prev` — seconds since this UID's previous transaction (0 for the first one)
- `delta_log_prev`     — log1p of `delta_seconds_prev` (compresses long tails)
- `uid_count_so_far`   — running count of transactions for this UID up to and including the current one


In [ ]:
# Mark each row's source set, then concat train + test into X_all
X_train['__source__'] = 'train'
X_test['__source__']  = 'test'

X_all = pd.concat([X_train, X_test], axis=0)
print(f'X_all rows: {len(X_all):,}  (train {len(X_train):,} + test {len(X_test):,})')


def add_time_gap_features(df):
    df = df.sort_values(['uid', 'DT'], kind='mergesort').copy()
    g = df.groupby('uid', sort=False)
    delta = g['TransactionDT'].diff().fillna(0).astype('float32')
    df['delta_seconds_prev'] = delta
    df['delta_log_prev']     = np.log1p(delta).astype('float32')
    df['uid_count_so_far']   = g.cumcount().astype('float32') + 1
    return df


X_all = add_time_gap_features(X_all)
feature_cols += ['delta_seconds_prev', 'delta_log_prev', 'uid_count_so_far']
print('feature columns now:', len(feature_cols))

# Quick sanity check: how many rows have a non-zero delta? (not the first txn of their UID)
share = (X_all['delta_seconds_prev'] > 0).mean()
print(f'rows that have a previous-UID-transaction reference: {share:.1%}')

# How many TEST rows benefit from the bridge (i.e. their previous-UID-transaction is in train)?
X_all = X_all.sort_values(['uid', 'DT'], kind='mergesort')
prev_source = X_all.groupby('uid', sort=False)['__source__'].shift(1)
bridge_mask = (X_all['__source__'] == 'test') & (prev_source == 'train')
n_bridge = bridge_mask.sum()
print(f'test rows whose immediate previous-UID-transaction is in TRAIN: {n_bridge:,} '
      f'({n_bridge / (X_all["__source__"] == "test").sum():.1%} of test rows)')


## 4. Imputation + standardization on the combined frame

The scaler is **fit on train rows only** to avoid leakage, but applied to both halves
of `X_all`. After this cell, every value in `feature_cols` is on a comparable scale
across train and test, which is critical for an LSTM.


In [ ]:
# Fill NaN with -1 (matching the XGBoost notebook's convention)
X_all[feature_cols] = X_all[feature_cols].fillna(-1).astype('float32')

# Fit scaler on train rows only, then transform both train and test rows in place
train_mask = (X_all['__source__'] == 'train')
scaler = StandardScaler()
X_all.loc[train_mask,  feature_cols] = scaler.fit_transform(
    X_all.loc[train_mask, feature_cols]).astype('float32')
X_all.loc[~train_mask, feature_cols] = scaler.transform(
    X_all.loc[~train_mask, feature_cols]).astype('float32')

print('train post-scale mean (first 5):',
      X_all.loc[train_mask, feature_cols[:5]].mean().round(3).to_list())
print('train post-scale std  (first 5):',
      X_all.loc[train_mask, feature_cols[:5]].std().round(3).to_list())


## 5. Build per-UID windows on the combined frame

Windows are built once on `X_all`, sorted by `(uid, DT)`. We then split the window
array into a train half and a test half by the `__source__` flag we added in section 3.

For a test row whose UID also has train transactions, the window will naturally
include those train transactions as the earlier timesteps — that's the bridge.

For a cold-start test UID (~78% of test rows), the window is still front-padded with
zeros, exactly as before.


In [ ]:
def build_uid_windows(df, feature_cols, window, uid_col='uid', time_col='DT'):
    """Build per-UID sliding windows on a frame sorted by (uid, DT).

    Returns
    -------
    X : (n_rows, window, n_features) float32
    idx : (n_rows,) — original index value (TransactionID) for each row of X, in window-order.
    """
    df_sorted = df.sort_values([uid_col, time_col], kind='mergesort')
    feat = df_sorted[feature_cols].to_numpy(dtype=np.float32)
    uids = df_sorted[uid_col].to_numpy()
    orig = df_sorted.index.to_numpy()

    n = len(df_sorted)
    f = len(feature_cols)
    X = np.zeros((n, window, f), dtype=np.float32)

    boundaries = np.flatnonzero(np.concatenate([[True], uids[1:] != uids[:-1]]))
    boundaries = np.append(boundaries, n)

    for b_start, b_end in zip(boundaries[:-1], boundaries[1:]):
        block = feat[b_start:b_end]
        for t in range(b_end - b_start):
            start = max(0, t - window + 1)
            seq = block[start:t + 1]
            X[b_start + t, -seq.shape[0]:] = seq
    return X, orig


# Build windows on the combined frame (train+test), sorted by (uid, DT)
print('Building combined windows on X_all (this is where the bridge is created)...')
t0 = time.time()
X_all_seq, all_order = build_uid_windows(X_all, feature_cols, WINDOW)
print(f'  shape={X_all_seq.shape}  built in {time.time()-t0:.1f}s')

# Recover the source flag in window-order, then split the array
source_in_order = X_all.loc[all_order, '__source__'].to_numpy()
train_pos = (source_in_order == 'train')
test_pos  = (source_in_order == 'test')

X_train_seq = X_all_seq[train_pos]
X_test_seq  = X_all_seq[test_pos]
train_order = all_order[train_pos]
test_order  = all_order[test_pos]

print(f'X_train_seq: {X_train_seq.shape}')
print(f'X_test_seq : {X_test_seq.shape}')

# Align targets and DT_M (used by GroupKFold) to the train window order
y_aligned    = y_train.loc[train_order].to_numpy(dtype=np.int8)
dt_m_aligned = X_all.loc[train_order, 'DT_M'].to_numpy()
print('y_aligned positives:', int(y_aligned.sum()),
      'rate:', y_aligned.mean().round(4))


In [ ]:
data = np.load("/Volumes/SandiskSSD/Developer/AI_Document/Financial_Fraud_Detection_Thesis/ieee-fraud-detection/split_data.npz")

X_train_seq = data["X_train_seq"]
L_train = data["L_train"]
X_test_seq = data["X_test_seq"]
L_test = data["L_test"]
train_order = data["train_order"]
test_order = data["test_order"]
train_pos = data["train_pos"]
test_pos = data["test_pos"]
y_aligned = data["y_aligned"]
dt_m_aligned = data["dt_m_aligned"]

## 6. Architecture choices — the full explanation

This section answers: *how many LSTM layers, what hidden size, when to use Dropout?*

### LSTM hidden units (32, 64, 128, 256, ...)

This is the size of the LSTM's hidden state vector. It controls the model's capacity:

| units | behavior |
|-------|----------|
| 32  | Tiny — fine for very simple patterns or small datasets. Will likely under-fit ~250 features. |
| 64  | A common sane default. Good first try. |
| 128 | More capacity, useful when each timestep has many features (your case). |
| 256+ | Lots of capacity. Needs lots of data and aggressive regularization (dropout, weight decay) or it will over-fit. |

**Rule of thumb for our setup** (window=5, ~250 features per step, ~590k training rows):
start with **128** and only go higher if validation AUC is still improving and not
diverging from training AUC.

### Number of LSTM layers (1 vs 2 vs 3+)

- **1 LSTM layer** — works for short sequences (≤10) and simple temporal patterns.
- **2 LSTM layers (stacked)** — can learn hierarchical patterns: the first layer sees raw timesteps, the second layer sees a sequence of "summaries" produced by the first. The first must use `return_sequences=True` so the second receives a sequence; the second uses `return_sequences=False` to emit a single vector for the dense head.
- **3+ layers** — rarely needed unless sequences are long (50+) and there's lots of data. For window=5, two layers is already plenty.

For our short sequences, **1 or 2 layers** is the right range.

### `return_sequences=True` vs `False`

- `True` returns the LSTM output at **every** timestep — shape `(batch, timesteps, units)`. Use this when the next layer is another recurrent layer.
- `False` returns only the **last** timestep — shape `(batch, units)`. Use this on the final recurrent layer when you want one prediction per sequence.

This is why stacked LSTMs look like:

```python
LSTM(128, return_sequences=True)   # passes a sequence to the next LSTM
LSTM(64,  return_sequences=False)  # collapses to a single vector
Dense(...)                         # head
```

### When to apply Dropout

Three places, three different effects:

1. **`dropout=` inside an LSTM layer** — drops elements of the *input* to the recurrent cell. Cheap, useful, doesn't break CuDNN GPU acceleration. Try **0.1–0.3**.
2. **`recurrent_dropout=` inside an LSTM layer** — drops elements of the recurrent state itself. Stronger regularization but **disables the CuDNN fast path**, so training becomes much slower on GPU. Use only if you really need it. Try **0.0–0.2**.
3. **`Dropout(p)` layer between Dense layers** — standard feed-forward dropout on the dense head. Try **0.3–0.5**. The 0.5 you saw in the slide is on the aggressive end.

For our highly imbalanced fraud problem (~3.5% positives), I'd start moderate
(`dropout=0.2`, then a `Dropout(0.3)` before the final dense). Aggressive dropout can
hurt the minority class because it adds noise that disproportionately drowns out the
small fraud signal.

### Loss function — important difference vs the weather notebook

The weather notebook used `MeanSquaredError` because it's a **regression** task
(predict a continuous temperature). Fraud detection is **binary classification**, so:

- Loss: `binary_crossentropy`
- Final layer: `Dense(1, activation='sigmoid')`
- Metric: `AUC` (matches the IEEE leaderboard)

If you copied `loss='mae'` or `loss='mse'` from the stock-price tutorial you'd be
training the wrong objective.

### Class imbalance handling

About 3.5% of rows are fraud. Two ways to handle:

1. **`class_weight={0: 1.0, 1: w}`** in `model.fit(...)` — re-weights the loss per sample. Set `w = (n_neg / n_pos)`.
2. **`sample_weight=`** for finer control.

Both work; class_weight is simpler.


In [ ]:
# 7. Build the model — MPS-aware variants
N_FEATURES = X_train_seq.shape[2]

def build_model_fast(window, n_features, units1=128, units2=64, dense=32,
                     drop_dense=0.3, lr=1e-3):
    """FAST_PATH variant.

    No Masking, no LSTM input dropout — these two settings are what allow
    TensorFlow on Metal (and CuDNN on NVIDIA) to use the fused recurrent kernel.
    Padded zero timesteps still flow through the LSTM, but with WINDOW=5 and
    standardised features the impact is tiny (a zero vector contributes very
    little to the gate activations) and the speedup is large.
    Regularisation is moved to the dense head.
    """
    model = Sequential([
        Input(shape=(window, n_features)),
        LSTM(units1, return_sequences=True),
        LSTM(units2, return_sequences=False),
        Dense(dense, activation='relu'),
        Dropout(drop_dense),
        Dense(1, activation='sigmoid', dtype='float32'),   # keep output fp32 even with mixed precision
    ])
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=[AUC(name='auc')],
    )
    return model


def build_model_strict(window, n_features, units1=128, units2=64, dense=32,
                       drop_rec=0.2, drop_dense=0.3, lr=1e-3):
    """Original variant with Masking + LSTM input dropout. Slower on Metal but
    arguably more rigorous about ignoring padded steps. Use for thesis comparison."""
    model = Sequential([
        Input(shape=(window, n_features)),
        Masking(mask_value=0.0),
        LSTM(units1, return_sequences=True, dropout=drop_rec),
        LSTM(units2, return_sequences=False, dropout=drop_rec),
        Dense(dense, activation='relu'),
        Dropout(drop_dense),
        Dense(1, activation='sigmoid', dtype='float32'),
    ])
    model.compile(
        optimizer=Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=[AUC(name='auc')],
    )
    return model


build_model = build_model_fast if FAST_PATH else build_model_strict
print(f'Using {"FAST" if FAST_PATH else "STRICT"} model variant')
model = build_model(WINDOW, N_FEATURES)
model.summary()


## 8. Train with strict expanding-window time validation

Replaces `GroupKFold(DT_M)` with a chronological expanding-window scheme:

- Sort the unique training months in ascending order.
- For each validation month after the first `MIN_TRAIN_MONTHS`, train on **all earlier
  months** and validate on that one month.
- Training data therefore strictly precedes validation data — no future-into-past leak.

With `MIN_TRAIN_MONTHS = 3` and the IEEE training months `{12, 13, 14, 15, 16, 17}`
(Dec 2017 – May 2018), this gives **3 folds**:

| Fold | Train months | Validate month |
|------|--------------|----------------|
| 0 | {12, 13, 14} | 15 |
| 1 | {12, 13, 14, 15} | 16 |
| 2 | {12, 13, 14, 15, 16} | 17 |

Months 12, 13, 14 are **never** in any validation fold — they only ever serve as
training history. So the OOF array is filled only for rows in months
`{15, 16, 17}` (about half of the training rows). The reported overall AUC is
computed only on those validated rows. To compare with your XGBoost OOF AUC, restrict
the XGB OOF to the same months — see notes after section 9.


In [ ]:
# 8. Train with strict expanding-window time validation
def expanding_month_folds(months_array, min_train_months=MIN_TRAIN_MONTHS):
    """Yield (valid_month, train_months, train_idx, valid_idx) tuples.

    The first validation fold uses the (min_train_months+1)-th month;
    later folds expand training to include all earlier months.
    """
    months = sorted(np.unique(months_array).tolist())
    valid_months = months[min_train_months:]
    for valid_month in valid_months:
        train_months = [m for m in months if m < valid_month]
        train_idx = np.flatnonzero(np.isin(months_array, train_months))
        valid_idx = np.flatnonzero(months_array == valid_month)
        yield (valid_month, train_months, train_idx, valid_idx)


# OOF only fills validated months; rest stays NaN
oof = np.full(len(X_train_seq), np.nan, dtype=np.float32)
fold_aucs = []
test_preds = np.zeros(len(X_test_seq), dtype=np.float32)


def make_train_ds(X, y, sw, batch):
    ds = tf.data.Dataset.from_tensor_slices((X, y, sw))
    ds = ds.shuffle(buffer_size=min(len(X), 100_000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch, drop_remainder=False).prefetch(AUTOTUNE)
    return ds


def make_eval_ds(X, y, batch):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    ds = ds.batch(batch, drop_remainder=False).prefetch(AUTOTUNE)
    return ds


def make_predict_ds(X, batch):
    ds = tf.data.Dataset.from_tensor_slices(X)
    ds = ds.batch(batch, drop_remainder=False).prefetch(AUTOTUNE)
    return ds


fold_specs = list(expanding_month_folds(dt_m_aligned, MIN_TRAIN_MONTHS))
print(f'Generated {len(fold_specs)} expanding-window folds:')
for vm, tm, ti, vi in fold_specs:
    print(f'   train months={tm} → validate month={vm}  '
          f'(train rows={len(ti):,}, valid rows={len(vi):,})')

for fold, (valid_month, train_months, idxT, idxV) in enumerate(fold_specs):
    print(f'\n=== Fold {fold}: train {train_months} → validate {valid_month} '
          f'(train={len(idxT):,}, valid={len(idxV):,}) ===')

    pos = y_aligned[idxT].sum()
    neg = len(idxT) - pos
    pos_w = float(neg / max(pos, 1))
    print(f'   class-1 weight: {pos_w:.2f}')

    sw_T = np.where(y_aligned[idxT] == 1, pos_w, 1.0).astype('float32')

    tf.keras.backend.clear_session()
    model = build_model(WINDOW, N_FEATURES)

    train_ds = make_train_ds(X_train_seq[idxT], y_aligned[idxT], sw_T, BATCH)
    val_ds   = make_eval_ds(X_train_seq[idxV], y_aligned[idxV], BATCH)

    callbacks = [
        EarlyStopping(monitor='val_auc', mode='max', patience=2,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5,
                          patience=1, min_lr=1e-5, verbose=1),
    ]

    t0 = time.time()
    hist = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks,
        verbose=2,
    )
    print(f'   fit time: {time.time()-t0:.1f}s')

    val_preds = model.predict(make_predict_ds(X_train_seq[idxV], BATCH), verbose=0).ravel()
    auc = roc_auc_score(y_aligned[idxV], val_preds)
    print(f'   fold AUC = {auc:.4f}  (validate month {valid_month})')
    fold_aucs.append((int(valid_month), float(auc)))
    oof[idxV] = val_preds

    test_preds += model.predict(make_predict_ds(X_test_seq, BATCH), verbose=0).ravel()

# Average test predictions across the folds we actually ran
n_folds = len(fold_specs)
if n_folds > 0:
    test_preds /= n_folds

# Overall OOF AUC restricted to validated rows only
validated = ~np.isnan(oof)
print(f'\n=== Validated rows: {validated.sum():,} of {len(oof):,} '
      f'({validated.mean():.1%}) ===')
overall_auc = roc_auc_score(y_aligned[validated], oof[validated])
print(f'=== LSTM OOF AUC (validated months only) = {overall_auc:.4f} ===')
print(f'   per-fold AUCs (month, auc): {fold_aucs}')


## 9. Save OOF and test predictions

The OOF CSV contains `NaN` for rows that were never in any validation fold (i.e. the
first `MIN_TRAIN_MONTHS` months — months 12, 13, 14 by default). Those rows aren't
"missing predictions" — they were used as pure training history and never held out.

To compare against your XGBoost OOF, restrict both arrays to the same months
(`DT_M ∈ {15, 16, 17}` by default). Example:

```python
import pandas as pd
from sklearn.metrics import roc_auc_score

xgb_oof  = pd.read_csv('../../csv_exported_files/oof_xgb_95.csv').set_index('TransactionID')
lstm_oof = pd.read_csv('oof_lstm.csv').set_index('TransactionID')

# DT_M is in your X_train_copy4 dataframe
dt_m = X_train_copy4['DT_M']
y    = y_train  # the binary label series indexed by TransactionID

mask = dt_m.isin([15, 16, 17])
ids  = dt_m.index[mask]

print('XGB  AUC on months 15-17 :', roc_auc_score(y.loc[ids], xgb_oof.loc[ids, 'oof']))
print('LSTM AUC on months 15-17 :', roc_auc_score(y.loc[ids], lstm_oof.loc[ids, 'oof_lstm']))
```

That gives you a directly comparable pair of numbers.


In [ ]:
oof_df = pd.DataFrame({'TransactionID': train_order, 'oof_lstm': oof})
oof_df = oof_df.set_index('TransactionID').reindex(X_train.index).reset_index()
oof_df.to_csv('oof_lstm.csv', index=False)

test_df = pd.DataFrame({'TransactionID': test_order, 'pred_lstm': test_preds})
test_df = test_df.set_index('TransactionID').reindex(X_test.index).reset_index()
test_df.to_csv('test_pred_lstm.csv', index=False)

print('Wrote oof_lstm.csv and test_pred_lstm.csv')


## 10. Ablations to try (for your thesis comparisons)

Once the baseline runs, try varying ONE thing at a time and log the OOF AUC. This is
the empirical content of your thesis chapter on architecture choices.

| variant | what to change | expected effect |
|---------|----------------|-----------------|
| Window size | 3 / 5 / 10 / 20 | longer window helps only for the ~13.6% of UIDs with enough history |
| Single vs stacked LSTM | `[LSTM(128)]` vs `[LSTM(128, ret_seq), LSTM(64)]` | stacked usually wins by 0.001–0.005 AUC |
| Hidden units | 64 / 128 / 256 | 128 is usually the sweet spot here |
| Dropout | 0.0 / 0.2 / 0.4 | higher dropout slows convergence; the right value depends on overfit gap |
| GRU vs LSTM | swap `LSTM(...)` for `GRU(...)` | GRU is faster, often within 0.001 AUC |
| Bidirectional | `Bidirectional(LSTM(...))` | helpful when future context is available — but in fraud the *current* row is the target, so this is sometimes counterproductive; try it |
| Time-gap features | with vs without `delta_*` | usually a small but real lift |

Recommended baseline (the one this notebook trains):

```
Masking(0.0)
LSTM(128, return_sequences=True, dropout=0.2)
LSTM(64,  return_sequences=False, dropout=0.2)
Dense(32, relu)
Dropout(0.3)
Dense(1, sigmoid)
loss=binary_crossentropy, optimizer=Adam(1e-3), metric=AUC
class_weight={0:1, 1: neg/pos}
```

## 11. Honest expectation

Even with a clean per-UID framing, beating XGBoost-with-UID-aggregations (your 0.9502)
is hard on this dataset, because:

- **77.9% of test rows are cold-start** (UID never seen in train) — the LSTM's per-UID memory adds nothing for those rows; predictions reduce to the current transaction's features alone.
- For the remaining ~22%, XGBoost was already exploiting the same UID-level signal via aggregations.

The thesis value is the *comparison itself* — show what each model learns, where the
LSTM helps (rows with rich UID history) and where it doesn't (cold starts), and discuss
the trade-off between the two paradigms. An ensemble of XGBoost + LSTM via OOF averaging
is also a reasonable contribution; even a small lift over 0.9502 is worth reporting.
